# Airline Sentiment Analysis with BiLSTM
**Author:** Rizwan  
**Model Type:** RNN (Bidirectional LSTM)  
**Dataset:** Twitter Airline Sentiment (Kaggle)  

This notebook documents the development and iterative improvement of a sentiment classifier using Bidirectional LSTM networks.

---
## QUICK START: Load Pre-Trained Model

**Skip all training cells** and jump directly to loading trained model:

1. **Run cells 1-9** (Setup, imports, data loading, preprocessing, vocabulary, model definition)
2. **Skip cells 10-30** (All training iterations - already done)
3. **Jump to NEW CELL below** to load your best model
4. **Continue from cell 31** (Visualizations, inference)

This saves 30-60 minutes of retraining time.

---

## 1. SETUP & IMPORTS

In [ ]:
# Fix PyTorch compatibility with Python 3.12
import subprocess
import sys

print("Reinstalling PyTorch for Python 3.12 compatibility...")
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', 
    '--upgrade', '--force-reinstall',
    'torch', 'torchvision', '--index-url', 
    'https://download.pytorch.org/whl/cu118'
])
print("PyTorch reinstalled successfully!")
print("\nIMPORTANT: Restart the kernel after running this cell!")

In [ ]:
# Install required packages
import subprocess
import sys

packages = [
    'torch',
    'torchvision', 
    'scikit-learn',
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn'
]

print("Installing required packages...")
for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"{package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
        print(f"{package} installed")

print("\nAll packages ready!")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.utils.class_weight import compute_class_weight

import re
import json
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. CONFIGURE PATHS

In [ ]:
print("="*80)
print("CONFIGURING PROJECT PATHS")
print("="*80)

# Configure paths (relative to project root)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
DATA_PATH = os.path.join(PROJECT_ROOT, 'data')
MODELS_PATH = os.path.join(PROJECT_ROOT, 'models', 'RNN')
OUTPUTS_PATH = os.path.join(PROJECT_ROOT, 'outputs', 'RNN')

# Create directories if they don't exist
os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)
os.makedirs(OUTPUTS_PATH, exist_ok=True)

print(f"\nFile save locations:")
print(f"  Data: {DATA_PATH}")
print(f"  Models: {MODELS_PATH}")
print(f"  Outputs: {OUTPUTS_PATH}")

## 3. LOAD DATASET

In [ ]:
print("="*80)
print("LOADING DATASET")
print("="*80)

def download_and_load_data():
    """
    Automatically download and load the Twitter Airline Sentiment dataset.
    Tries multiple sources: local file, Kaggle, and GitHub backup.
    """
    import subprocess
    import glob
    
    csv_path = os.path.join(DATA_PATH, 'Tweets.csv')
    
    # Check if file already exists locally
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            if df.shape[0] > 10000:
                print(f"Loaded from local file: {df.shape[0]} rows")
                return df
        except Exception as e:
            print(f"Error reading local file: {e}")
    
    # Try downloading from Kaggle
    print("Dataset not found locally. Attempting to download...")
    try:
        print("\n[1/2] Trying Kaggle download...")
        subprocess.run(['pip', 'install', '-q', 'kagglehub'], 
                      capture_output=True, timeout=30)
        import kagglehub
        
        path = kagglehub.dataset_download("crowdflower/twitter-airline-sentiment")
        print(f"Downloaded to: {path}")
        
        csv_files = glob.glob(f"{path}/*.csv")
        if csv_files:
            source_file = csv_files[0]
            df = pd.read_csv(source_file)
            
            # Copy to project data folder
            df.to_csv(csv_path, index=False)
            print(f"Loaded {df.shape[0]} rows from Kaggle")
            print(f"Saved to: {csv_path}")
            return df
    except Exception as e:
        print(f"Kaggle download failed: {e}")
    
    # Fallback to GitHub source
    try:
        print("\n[2/2] Trying GitHub backup source...")
        github_url = 'https://raw.githubusercontent.com/koushikjanarth/Airline-Tweet-Sentiment-Analysis/master/Tweets.csv'
        
        # Try using urllib (built-in)
        import urllib.request
        urllib.request.urlretrieve(github_url, csv_path)
        
        if os.path.exists(csv_path) and os.path.getsize(csv_path) > 1000000:
            df = pd.read_csv(csv_path)
            print(f"Loaded {df.shape[0]} rows from GitHub")
            print(f"Saved to: {csv_path}")
            return df
    except Exception as e:
        print(f"GitHub download failed: {e}")
    
    # If all methods fail
    print("\n Automatic download failed.")
    print("\nManual download instructions:")
    print("1. Go to: https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment")
    print("2. Download Tweets.csv")
    print(f"3. Place it in: {DATA_PATH}")
    raise FileNotFoundError(f"Could not download dataset. Please download manually to {csv_path}")

# Load the dataset
df = download_and_load_data()

print(f"\nDataset Info:")
print(f"  Total samples: {len(df)}")
print(f"  Sentiment distribution:")
print(df['airline_sentiment'].value_counts())
print(f"\n  Sample texts:")
for idx in range(min(3, len(df))):
    print(f"    [{df['airline_sentiment'].iloc[idx]}] {df['text'].iloc[idx][:60]}...")

## 4. TEXT PREPROCESSING & DATA SPLITTING

In [ ]:
print("\n" + "="*80)
print("TEXT PREPROCESSING")
print("="*80)

def clean_text(text):
    # Convert to lowercase
    text = str(text).lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove @mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtag symbol
    text = re.sub(r'#', '', text)
    # Remove special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Clean all text data
df['text'] = df['text'].apply(clean_text)

print(f"Sample cleaned texts:")
for idx in range(min(3, len(df))):
    print(f"  {df['text'].iloc[idx][:70]}...")

print("\n" + "="*80)
print("TRAIN/VAL/TEST SPLIT")
print("="*80)

# SPLIT DATA FIRST (before building vocabulary)
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['text'].values, df['airline_sentiment'].values,
    test_size=0.3, random_state=42, stratify=df['airline_sentiment'].values
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels,
    test_size=0.5, random_state=42, stratify=temp_labels
)

print(f"Train set: {len(train_texts)} samples")
print(f"Val set: {len(val_texts)} samples")
print(f"Test set: {len(test_texts)} samples")

## 5. VOCABULARY BUILDING

In [ ]:
print("\n" + "="*80)
print("VOCABULARY BUILDING (FROM TRAIN SET ONLY)")
print("="*80)

class Vocabulary:
    def __init__(self, min_freq=2):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.word_freq = {}
        self.min_freq = min_freq

    def build_vocab(self, texts):
        # Count word frequencies
        for text in texts:
            for word in text.split():
                self.word_freq[word] = self.word_freq.get(word, 0) + 1

        # Build word to index mapping
        idx = 2
        for word, freq in self.word_freq.items():
            if freq >= self.min_freq:
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                idx += 1

        print(f"Vocabulary size: {len(self.word2idx)} words")

    def encode(self, text):
        return [self.word2idx.get(word, 1) for word in text.split()]

# Build vocabulary from TRAIN SET ONLY (no leakage)
vocab = Vocabulary(min_freq=2)
vocab.build_vocab(train_texts)
print("Vocabulary built from training data only (no validation/test leakage)")

## 6. PYTORCH DATASET & DATALOADER

In [ ]:
print("\n" + "="*80)
print("CREATING PYTORCH DATASET & DATALOADER")
print("="*80)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=50):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

        self.label2idx = {
            'positive': 0,
            'negative': 1,
            'neutral': 2
        }
        self.idx2label = {v: k for k, v in self.label2idx.items()}

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoded = self.vocab.encode(text)

        if len(encoded) < self.max_len:
            encoded = encoded + [0] * (self.max_len - len(encoded))
        else:
            encoded = encoded[:self.max_len]

        return {
            'text': torch.tensor(encoded, dtype=torch.long),
            'label': torch.tensor(self.label2idx[label], dtype=torch.long)
        }

# Create dataset objects
train_dataset = SentimentDataset(train_texts, train_labels, vocab, max_len=50)
val_dataset = SentimentDataset(val_texts, val_labels, vocab, max_len=50)
test_dataset = SentimentDataset(test_texts, test_labels, vocab, max_len=50)

# Create data loaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders created with batch size: {BATCH_SIZE}")

## 7. MODEL ARCHITECTURE - BiLSTM

---
### OPTION: Load Pre-Trained Model (Skip Training)

**Run this cell instead of training cells 10-30 to load your best model:**

In [ ]:
print("\n" + "="*80)
print("LOADING PRE-TRAINED MODELS (SKIP TRAINING)")
print("="*80)

import os
import pickle

# Check which trained models exist
trained_models = {
    'iter1': f'{MODELS_PATH}/best_sentiment_model_iter1.pt',
    'iter2': f'{MODELS_PATH}/best_sentiment_model_iter2.pt',
    'iter3': f'{MODELS_PATH}/best_sentiment_model_iter3.pt',
    'iter4': f'{MODELS_PATH}/best_sentiment_model_iter4.pt',
    'iter5': f'{MODELS_PATH}/best_sentiment_model_iter5.pt',
    'final': f'{MODELS_PATH}/sentiment_classifier_final.pt'
}

print("\nAvailable trained models:")
available_models = {}
for name, path in trained_models.items():
    if os.path.exists(path):
        file_size = os.path.getsize(path) / (1024 * 1024)  # MB
        print(f"  {name}: {path} ({file_size:.2f} MB)")
        available_models[name] = path
    else:
        print(f"  {name}: Not found (needs training)")

if not available_models:
    print("\nNo trained models found. You need to train first or check paths.")
    print(f"Expected location: {MODELS_PATH}")
else:
    # Load the best model (Iteration 5 or final)
    best_model_name = 'final' if 'final' in available_models else 'iter5'
    if best_model_name not in available_models:
        best_model_name = list(available_models.keys())[-1]
    
    print(f"\nLoading best model: {best_model_name}")
    
    # Create model architecture (Iteration 5 config)
    vocab_size = len(vocab.word2idx)
    model_iter5 = SentimentLSTM(
        vocab_size=vocab_size,
        embedding_dim=150,
        hidden_dim=160,
        output_dim=3,
        n_layers=3,
        bidirectional=True,
        dropout=0.4
    ).to(device)
    
    # Load trained weights
    model_iter5.load_state_dict(torch.load(available_models[best_model_name]))
    model_iter5.eval()
    
    print(f"Model loaded successfully")
    print(f"  Parameters: {sum(p.numel() for p in model_iter5.parameters()):,}")
    print(f"  Device: {device}")
    
    # Quick validation
    criterion_iter5 = nn.CrossEntropyLoss()
    test_loss_iter5, test_acc_iter5, test_preds_iter5, test_labels_np_iter5 = evaluate(
        model_iter5, test_loader, criterion_iter5, device
    )
    
    print(f"\nLoaded Model Performance:")
    print(f"  Test Accuracy: {test_acc_iter5:.4f}")
    print(f"  Test Loss: {test_loss_iter5:.4f}")
    
    print("\nReady for inference. Skip to Section 16 (Inference) or 17 (Summary)")
    print("="*80)

In [ ]:
print("\n" + "="*80)
print("BUILDING LSTM SENTIMENT CLASSIFIER")
print("="*80)

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, hidden_dim=128,
                 output_dim=3, n_layers=2, bidirectional=True, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True
        )

        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim

        self.fc1 = nn.Linear(lstm_output_dim, 64)
        self.fc2 = nn.Linear(64, output_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        embedded = self.dropout(self.embedding(text))

        lstm_out, (hidden, cell) = self.lstm(embedded)

        hidden = self.dropout(hidden)

        if self.lstm.bidirectional:
            hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            hidden = hidden[-1]

        out = self.fc1(hidden)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)

        return out

# Initialize first model
vocab_size = len(vocab.word2idx)
model = SentimentLSTM(
    vocab_size=vocab_size,
    embedding_dim=100,
    hidden_dim=128,
    output_dim=3,
    n_layers=2,
    bidirectional=True,
    dropout=0.3
).to(device)

print(f"Model created successfully")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {device}")

## 8. TRAINING FUNCTIONS

In [ ]:
print("\n" + "="*80)
print("CONFIGURING TRAINING FUNCTIONS")
print("="*80)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

def train_epoch(model, iterator, optimizer, criterion, device):
    model.train()
    epoch_loss = 0
    all_preds = []
    all_labels = []

    for batch in iterator:
        text = batch['text'].to(device)
        labels = batch['label'].to(device)

        predictions = model(text)
        loss = criterion(predictions, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()
        all_preds.extend(predictions.argmax(dim=1).cpu().detach().numpy())
        all_labels.extend(labels.cpu().detach().numpy())

    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_loss = epoch_loss / len(iterator)

    return epoch_loss, epoch_acc

def evaluate(model, iterator, criterion, device):
    model.eval()
    epoch_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in iterator:
            text = batch['text'].to(device)
            labels = batch['label'].to(device)

            predictions = model(text)
            loss = criterion(predictions, labels)

            epoch_loss += loss.item()
            all_preds.extend(predictions.argmax(dim=1).cpu().detach().numpy())
            all_labels.extend(labels.cpu().detach().numpy())

    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_loss = epoch_loss / len(iterator)

    return epoch_loss, epoch_acc, all_preds, all_labels

print(f"Training functions defined")

## 9. ITERATION 1: BASELINE MODEL

In [ ]:
print("\n" + "="*80)
print("ITERATION 1: BASELINE MODEL (2 LSTM layers, 128 hidden, dropout=0.3)")
print("="*80)

NUM_EPOCHS = 15
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'best_val_loss': float('inf')
}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_loss < history['best_val_loss']:
        history['best_val_loss'] = val_loss
        save_path = f'{MODELS_PATH}/best_sentiment_model_iter1.pt'
        torch.save(model.state_dict(), save_path)

    if (epoch + 1) % 3 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

# Load the best model checkpoint
best_model_path = f'{MODELS_PATH}/best_sentiment_model_iter1.pt'
model.load_state_dict(torch.load(best_model_path))

# Test on test set
test_loss, test_acc, test_preds, test_labels_np = evaluate(model, test_loader, criterion, device)

print(f"\n{'='*80}")
print(f"ITERATION 1 RESULTS (TEST SET)")
print(f"{'='*80}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

labels_list = ['positive', 'negative', 'neutral']
print(f"\nDetailed Classification Report:")
print(classification_report(test_labels_np, test_preds, target_names=labels_list))

cm_iter1 = confusion_matrix(test_labels_np, test_preds)
print(f"\nConfusion Matrix:")
print(cm_iter1)

print(f"\nBest model saved to: {best_model_path}")

## 10. ITERATION 2: IMPROVED MODEL

In [ ]:
print("\n" + "="*80)
print("ITERATION 2: IMPROVED MODEL")
print("Changes: Deeper embedding (150), larger hidden (160), 3 LSTM layers")
print("="*80)

# Create improved model architecture
model_iter2 = SentimentLSTM(
    vocab_size=vocab_size,
    embedding_dim=150,
    hidden_dim=160,
    output_dim=3,
    n_layers=3,
    bidirectional=True,
    dropout=0.35
).to(device)

optimizer_iter2 = optim.Adam(model_iter2.parameters(), lr=0.001, weight_decay=1e-5)

history_iter2 = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'best_val_loss': float('inf')
}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model_iter2, train_loader, optimizer_iter2, criterion, device)
    val_loss, val_acc, _, _ = evaluate(model_iter2, val_loader, criterion, device)

    history_iter2['train_loss'].append(train_loss)
    history_iter2['train_acc'].append(train_acc)
    history_iter2['val_loss'].append(val_loss)
    history_iter2['val_acc'].append(val_acc)

    if val_loss < history_iter2['best_val_loss']:
        history_iter2['best_val_loss'] = val_loss
        save_path = f'{MODELS_PATH}/best_sentiment_model_iter2.pt'
        torch.save(model_iter2.state_dict(), save_path)

    if (epoch + 1) % 3 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

# Load best checkpoint
best_model_path_iter2 = f'{MODELS_PATH}/best_sentiment_model_iter2.pt'
model_iter2.load_state_dict(torch.load(best_model_path_iter2))
test_loss_iter2, test_acc_iter2, test_preds_iter2, test_labels_np_iter2 = evaluate(
    model_iter2, test_loader, criterion, device
)

print(f"\n{'='*80}")
print(f"ITERATION 2 RESULTS (TEST SET)")
print(f"{'='*80}")
print(f"Test Accuracy: {test_acc_iter2:.4f}")
print(f"Test Loss: {test_loss_iter2:.4f}")

print(f"\nDetailed Classification Report:")
print(classification_report(test_labels_np_iter2, test_preds_iter2, target_names=labels_list))

cm_iter2 = confusion_matrix(test_labels_np_iter2, test_preds_iter2)
print(f"\nConfusion Matrix:")
print(cm_iter2)

print(f"\nBest model saved to: {best_model_path_iter2}")

## 11. ITERATION 3: EARLY STOPPING

### Early Stopping Helper Class

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0

    def step(self, val_loss):
        """
        Returns True if training should stop.
        """
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

### Train Iteration 3

In [ ]:
print("\n" + "="*80)
print("ITERATION 3: EARLY STOPPING (same as Iteration 2 architecture)")
print("="*80)

model_iter3 = SentimentLSTM(
    vocab_size=vocab_size,
    embedding_dim=150,
    hidden_dim=160,
    output_dim=3,
    n_layers=3,
    bidirectional=True,
    dropout=0.35
).to(device)

optimizer_iter3 = optim.Adam(model_iter3.parameters(), lr=0.001, weight_decay=1e-5)

PATIENCE = 3
MIN_DELTA = 0.0
early_stopper = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA)

NUM_EPOCHS = 30
best_model_path_iter3 = f"{MODELS_PATH}/best_sentiment_model_iter3.pt"

history_iter3 = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "best_val_loss": float("inf"),
    "best_epoch": None
}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model_iter3, train_loader, optimizer_iter3, criterion, device)
    val_loss, val_acc, _, _ = evaluate(model_iter3, val_loader, criterion, device)

    history_iter3["train_loss"].append(train_loss)
    history_iter3["train_acc"].append(train_acc)
    history_iter3["val_loss"].append(val_loss)
    history_iter3["val_acc"].append(val_acc)

    if val_loss < history_iter3["best_val_loss"]:
        history_iter3["best_val_loss"] = val_loss
        history_iter3["best_epoch"] = epoch + 1
        torch.save(model_iter3.state_dict(), best_model_path_iter3)

    if epoch == 0 or (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch+1:2d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if early_stopper.step(val_loss):
        print(f"\nEarly stopping triggered at epoch {epoch+1}.")
        print(f"Best val loss: {history_iter3['best_val_loss']:.4f} at epoch {history_iter3['best_epoch']}.")
        break

model_iter3.load_state_dict(torch.load(best_model_path_iter3))

test_loss_iter3, test_acc_iter3, test_preds_iter3, test_labels_np_iter3 = evaluate(
    model_iter3, test_loader, criterion, device
)

print(f"\n{'='*80}")
print("ITERATION 3 RESULTS (TEST SET) - EARLY STOPPING")
print(f"{'='*80}")
print(f"Best Epoch (by val loss): {history_iter3['best_epoch']}")
print(f"Test Accuracy: {test_acc_iter3:.4f}")
print(f"Test Loss: {test_loss_iter3:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(test_labels_np_iter3, test_preds_iter3, target_names=labels_list))

cm_iter3 = confusion_matrix(test_labels_np_iter3, test_preds_iter3)
print("\nConfusion Matrix:")
print(cm_iter3)

print(f"\nBest model saved to: {best_model_path_iter3}")

## 12. ITERATION 4: CLASS-WEIGHTED LOSS

### Compute Class Weights

In [ ]:
print("\n" + "="*80)
print("ITERATION 4: CLASS-WEIGHTED LOSS (Handling Class Imbalance)")
print("="*80)

label2idx = {
    'positive': 0,
    'negative': 1,
    'neutral': 2
}

y_train_int = np.array([label2idx[label] for label in train_labels])

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=y_train_int
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print("Class weights computed from training set:")
for i, w in enumerate(class_weights):
    print(f"  Class {labels_list[i]}: weight = {w:.4f}")

### Build and Train Iteration 4

In [ ]:
model_iter4 = SentimentLSTM(
    vocab_size=vocab_size,
    embedding_dim=150,
    hidden_dim=160,
    output_dim=3,
    n_layers=3,
    bidirectional=True,
    dropout=0.5
).to(device)

optimizer_iter4 = optim.Adam(model_iter4.parameters(), lr=0.001, weight_decay=1e-5)
criterion_iter4 = nn.CrossEntropyLoss(weight=class_weights_tensor)

print("Iteration 4 model + class-weighted loss initialized.")
print(f"Total parameters (Iter 4): {sum(p.numel() for p in model_iter4.parameters()):,}")

In [ ]:
print("\n" + "="*80)
print("ITERATION 4: TRAINING (Class-Weighted + Early Stopping)")
print("="*80)

history_iter4 = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_loss_iter4 = float('inf')
best_epoch_iter4 = 0
patience = 3
patience_counter = 0

NUM_EPOCHS_ITER4 = 12

for epoch in range(1, NUM_EPOCHS_ITER4 + 1):
    train_loss, train_acc = train_epoch(
        model_iter4, train_loader, optimizer_iter4, criterion_iter4, device
    )

    val_loss, val_acc, _, _ = evaluate(
        model_iter4, val_loader, criterion_iter4, device
    )

    history_iter4['train_loss'].append(train_loss)
    history_iter4['train_acc'].append(train_acc)
    history_iter4['val_loss'].append(val_loss)
    history_iter4['val_acc'].append(val_acc)

    print(f"Epoch {epoch:2d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_loss < best_val_loss_iter4:
        best_val_loss_iter4 = val_loss
        best_epoch_iter4 = epoch
        torch.save(model_iter4.state_dict(), f"{MODELS_PATH}/best_sentiment_model_iter4.pt")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch}.")
            break

print(f"\nBest val loss (Iter 4): {best_val_loss_iter4:.4f} at epoch {best_epoch_iter4}.")

### Iteration 4 Test Evaluation

In [ ]:
model_iter4.load_state_dict(torch.load(f"{MODELS_PATH}/best_sentiment_model_iter4.pt"))

test_loss_iter4, test_acc_iter4, test_preds_iter4, test_labels_np_iter4 = evaluate(
    model_iter4, test_loader, criterion_iter4, device
)

cm_iter4 = confusion_matrix(test_labels_np_iter4, test_preds_iter4)

print("\n" + "="*80)
print("ITERATION 4 RESULTS (TEST SET) - CLASS-WEIGHTED LOSS")
print("="*80)
print(f"Best Epoch (by val loss): {best_epoch_iter4}")
print(f"Test Accuracy: {test_acc_iter4:.4f}")
print(f"Test Loss: {test_loss_iter4:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(test_labels_np_iter4, test_preds_iter4, target_names=labels_list))

print("\nConfusion Matrix:")
print(cm_iter4)

## 13. ITERATION 5 (FINAL): OPTIMIZED BiLSTM

### Build Model with LR Scheduler

In [ ]:
print("\n" + "="*80)
print("ITERATION 5 (FINAL): Optimised BiLSTM + LR Scheduler + Early Stopping")
print("="*80)

model_iter5 = SentimentLSTM(
    vocab_size=vocab_size,
    embedding_dim=150,
    hidden_dim=160,
    output_dim=3,
    n_layers=3,
    bidirectional=True,
    dropout=0.4
).to(device)

criterion_iter5 = nn.CrossEntropyLoss()

optimizer_iter5 = optim.Adam(
    model_iter5.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

scheduler_iter5 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_iter5,
    mode='min',
    factor=0.5,
    patience=1
)

print("Iteration 5 model initialised.")
print(f"Total parameters (Iter 5): {sum(p.numel() for p in model_iter5.parameters()):,}")

### Train Iteration 5

In [ ]:
history_iter5 = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'lr': []
}

best_val_loss_iter5 = float('inf')
best_epoch_iter5 = 0
patience = 3
patience_counter = 0

NUM_EPOCHS_ITER5 = 15
best_path_iter5 = f"{MODELS_PATH}/best_sentiment_model_iter5.pt"

for epoch in range(1, NUM_EPOCHS_ITER5 + 1):
    train_loss, train_acc = train_epoch(
        model_iter5, train_loader, optimizer_iter5, criterion_iter5, device
    )
    val_loss, val_acc, _, _ = evaluate(
        model_iter5, val_loader, criterion_iter5, device
    )

    scheduler_iter5.step(val_loss)

    current_lr = optimizer_iter5.param_groups[0]['lr']
    history_iter5['lr'].append(current_lr)

    history_iter5['train_loss'].append(train_loss)
    history_iter5['train_acc'].append(train_acc)
    history_iter5['val_loss'].append(val_loss)
    history_iter5['val_acc'].append(val_acc)

    print(f"Epoch {epoch:2d} | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_loss < best_val_loss_iter5:
        best_val_loss_iter5 = val_loss
        best_epoch_iter5 = epoch
        torch.save(model_iter5.state_dict(), best_path_iter5)
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch}.")
            break

print(f"\nBest val loss (Iter 5): {best_val_loss_iter5:.4f} at epoch {best_epoch_iter5}.")
print(f"Saved best Iter 5 model to: {best_path_iter5}")

### Iteration 5 Test Evaluation

In [ ]:
model_iter5.load_state_dict(torch.load(best_path_iter5))

test_loss_iter5, test_acc_iter5, test_preds_iter5, test_labels_np_iter5 = evaluate(
    model_iter5, test_loader, criterion_iter5, device
)

print(f"\n{'='*80}")
print("ITERATION 5 RESULTS (TEST SET) - OPTIMISED BiLSTM")
print(f"{'='*80}")
print(f"Best Epoch (by val loss): {best_epoch_iter5}")
print(f"Test Accuracy: {test_acc_iter5:.4f}")
print(f"Test Loss: {test_loss_iter5:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(test_labels_np_iter5, test_preds_iter5, target_names=labels_list))

cm_iter5 = confusion_matrix(test_labels_np_iter5, test_preds_iter5)
print("\nConfusion Matrix:")
print(cm_iter5)

## 14. VISUALIZATIONS

### Training Curves Comparison (All 5 Iterations)

In [ ]:
print("\n" + "="*80)
print("GENERATING VISUALIZATIONS (Iterations 1–5)")
print("="*80)

fig, axes = plt.subplots(2, 5, figsize=(30, 9))
fig.suptitle('Sentiment Classifier - Training Progress Comparison (Iter 1–5)', fontsize=16, fontweight='bold')

# Loss plots
axes[0, 0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Validation', linewidth=2)
axes[0, 0].set_title('Iter 1: Loss', fontweight='bold')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(history_iter2['train_loss'], label='Train', linewidth=2)
axes[0, 1].plot(history_iter2['val_loss'], label='Validation', linewidth=2)
axes[0, 1].set_title('Iter 2: Loss', fontweight='bold')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

axes[0, 2].plot(history_iter3['train_loss'], label='Train', linewidth=2)
axes[0, 2].plot(history_iter3['val_loss'], label='Validation', linewidth=2)
axes[0, 2].set_title('Iter 3: Loss (Early Stop)', fontweight='bold')
axes[0, 2].set_xlabel('Epoch'); axes[0, 2].set_ylabel('Loss')
axes[0, 2].legend(); axes[0, 2].grid(alpha=0.3)

axes[0, 3].plot(history_iter4['train_loss'], label='Train', linewidth=2)
axes[0, 3].plot(history_iter4['val_loss'], label='Validation', linewidth=2)
axes[0, 3].set_title('Iter 4: Loss (Weighted)', fontweight='bold')
axes[0, 3].set_xlabel('Epoch'); axes[0, 3].set_ylabel('Loss')
axes[0, 3].legend(); axes[0, 3].grid(alpha=0.3)

axes[0, 4].plot(history_iter5['train_loss'], label='Train', linewidth=2)
axes[0, 4].plot(history_iter5['val_loss'], label='Validation', linewidth=2)
axes[0, 4].set_title('Iter 5: Loss (Final)', fontweight='bold')
axes[0, 4].set_xlabel('Epoch'); axes[0, 4].set_ylabel('Loss')
axes[0, 4].legend(); axes[0, 4].grid(alpha=0.3)

# Accuracy plots
axes[1, 0].plot(history['train_acc'], label='Train', linewidth=2)
axes[1, 0].plot(history['val_acc'], label='Validation', linewidth=2)
axes[1, 0].set_title('Iter 1: Accuracy', fontweight='bold')
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(history_iter2['train_acc'], label='Train', linewidth=2)
axes[1, 1].plot(history_iter2['val_acc'], label='Validation', linewidth=2)
axes[1, 1].set_title('Iter 2: Accuracy', fontweight='bold')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

axes[1, 2].plot(history_iter3['train_acc'], label='Train', linewidth=2)
axes[1, 2].plot(history_iter3['val_acc'], label='Validation', linewidth=2)
axes[1, 2].set_title('Iter 3: Accuracy (Early Stop)', fontweight='bold')
axes[1, 2].set_xlabel('Epoch'); axes[1, 2].set_ylabel('Accuracy')
axes[1, 2].legend(); axes[1, 2].grid(alpha=0.3)

axes[1, 3].plot(history_iter4['train_acc'], label='Train', linewidth=2)
axes[1, 3].plot(history_iter4['val_acc'], label='Validation', linewidth=2)
axes[1, 3].set_title('Iter 4: Accuracy (Weighted)', fontweight='bold')
axes[1, 3].set_xlabel('Epoch'); axes[1, 3].set_ylabel('Accuracy')
axes[1, 3].legend(); axes[1, 3].grid(alpha=0.3)

axes[1, 4].plot(history_iter5['train_acc'], label='Train', linewidth=2)
axes[1, 4].plot(history_iter5['val_acc'], label='Validation', linewidth=2)
axes[1, 4].set_title('Iter 5: Accuracy (Final)', fontweight='bold')
axes[1, 4].set_xlabel('Epoch'); axes[1, 4].set_ylabel('Accuracy')
axes[1, 4].legend(); axes[1, 4].grid(alpha=0.3)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

curves_path = f'{OUTPUTS_PATH}/training_curves_iter1_5.png'
plt.savefig(curves_path, dpi=300, bbox_inches='tight')
print(f"Saved: {curves_path}")
plt.show()

### Confusion Matrices (All 5 Iterations)

In [ ]:
print("\n" + "="*80)
print("CONFUSION MATRICES (Iterations 1–5)")
print("="*80)

fig, axes = plt.subplots(1, 5, figsize=(30, 5))
fig.suptitle('Confusion Matrices - Test Set Performance (Iter 1–5)', fontsize=14, fontweight='bold')

sns.heatmap(cm_iter1, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_list, yticklabels=labels_list,
            cbar=True, ax=axes[0])
axes[0].set_title(f'Iter 1 (Acc: {test_acc:.4f})', fontweight='bold')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Pred')

sns.heatmap(cm_iter2, annot=True, fmt='d', cmap='Oranges',
            xticklabels=labels_list, yticklabels=labels_list,
            cbar=True, ax=axes[1])
axes[1].set_title(f'Iter 2 (Acc: {test_acc_iter2:.4f})', fontweight='bold')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Pred')

sns.heatmap(cm_iter3, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels_list, yticklabels=labels_list,
            cbar=True, ax=axes[2])
axes[2].set_title(f'Iter 3 (Acc: {test_acc_iter3:.4f})', fontweight='bold')
axes[2].set_ylabel('True'); axes[2].set_xlabel('Pred')

sns.heatmap(cm_iter4, annot=True, fmt='d', cmap='Purples',
            xticklabels=labels_list, yticklabels=labels_list,
            cbar=True, ax=axes[3])
axes[3].set_title(f'Iter 4 (Acc: {test_acc_iter4:.4f})', fontweight='bold')
axes[3].set_ylabel('True'); axes[3].set_xlabel('Pred')

sns.heatmap(cm_iter5, annot=True, fmt='d', cmap='Greys',
            xticklabels=labels_list, yticklabels=labels_list,
            cbar=True, ax=axes[4])
axes[4].set_title(f'Iter 5 (Acc: {test_acc_iter5:.4f})', fontweight='bold')
axes[4].set_ylabel('True'); axes[4].set_xlabel('Pred')

plt.tight_layout(rect=[0, 0.03, 1, 0.90])

cm_path = f'{OUTPUTS_PATH}/confusion_matrices_iter1_5.png'
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
print(f"Saved: {cm_path}")
plt.show()

## 15. MISCLASSIFICATION ANALYSIS

In [ ]:
print("\n" + "="*80)
print("MISCLASSIFICATION ANALYSIS")
print("="*80)

def analyze_misclassifications(texts, true_labels, pred_labels, label_map, n_samples=5):
    misclassified = []

    for idx, (text, true_label, pred_label) in enumerate(zip(texts, true_labels, pred_labels)):
        if true_label != pred_label:
            misclassified.append({
                'text': text,
                'true_label': label_map[true_label],
                'pred_label': label_map[pred_label]
            })

    print(f"\nTotal misclassifications: {len(misclassified)} / {len(texts)} "
          f"({len(misclassified)/len(texts)*100:.2f}%)")
    print(f"\nTop {n_samples} misclassified examples:")

    for i, item in enumerate(misclassified[:n_samples], 1):
        print(f"\n  {i}. TRUE: {item['true_label'].upper()} → PRED: {item['pred_label'].upper()}")
        print(f"     Text: \"{item['text'][:80]}...\"")

label_map = {0: 'positive', 1: 'negative', 2: 'neutral'}

print("\n--- ITERATION 1 MISCLASSIFICATIONS ---")
analyze_misclassifications(test_texts, test_labels_np, test_preds, label_map, n_samples=5)

print("\n--- ITERATION 5 (FINAL) MISCLASSIFICATIONS ---")
analyze_misclassifications(test_texts, test_labels_np_iter5, test_preds_iter5, label_map, n_samples=5)

## 16. INFERENCE FUNCTION & DEMO

In [ ]:
print("\n" + "="*80)
print("INFERENCE FUNCTION & TEST WITH REAL SAMPLES")
print("="*80)

def predict_sentiment(text, model, vocab, device, label_map={0: 'positive', 1: 'negative', 2: 'neutral'}):
    model.eval()

    cleaned_text = clean_text(text)
    encoded = vocab.encode(cleaned_text)

    if len(encoded) < 50:
        encoded = encoded + [0] * (50 - len(encoded))
    else:
        encoded = encoded[:50]

    text_tensor = torch.tensor([encoded], dtype=torch.long).to(device)

    with torch.no_grad():
        output = model(text_tensor)
        probabilities = torch.softmax(output, dim=1)
        predicted_class = output.argmax(dim=1).item()

    return {
        'sentiment': label_map[predicted_class],
        'confidence': probabilities[0][predicted_class].item(),
        'probabilities': {
            label_map[i]: probabilities[0][i].item()
            for i in range(3)
        }
    }

# Test with real examples from test set
print("\nUsing Iteration 5 (Final Model) with real test samples:\n")

np.random.seed(42)
sample_indices = np.random.choice(len(test_texts), 5, replace=False)

for idx in sample_indices:
    text = test_texts[idx]
    true_label = test_labels[idx]
    result = predict_sentiment(text, model_iter5, vocab, device)

    print(f"True Label: {true_label}")
    print(f"Input: \"{text}\"")
    print(f"Predicted Sentiment: {result['sentiment'].upper()} (confidence: {result['confidence']:.4f})")
    print(f"  → Positive: {result['probabilities']['positive']:.4f}")
    print(f"  → Negative: {result['probabilities']['negative']:.4f}")
    print(f"  → Neutral:  {result['probabilities']['neutral']:.4f}")
    print()

### Interactive Demo

In [ ]:
# Simple interactive demo
user_text = input("Enter passenger message: ")
result = predict_sentiment(user_text, model_iter5, vocab, device)
print("Predicted Sentiment:", result['sentiment'])
print("Confidence:", result['confidence'])

## 17. FINAL SUMMARY & MODEL SAVING

In [ ]:
print("\n" + "="*80)
print("FINAL MODEL SUMMARY & COMPARISON")
print("="*80)

comparison_data = {
    'Metric': ['Test Accuracy', 'Test Loss', 'Architecture', 'Epochs Trained'],
    'Iteration 1': [
        f'{test_acc:.4f}',
        f'{test_loss:.4f}',
        '2 LSTM (128)',
        '15'
    ],
    'Iteration 5 (Final)': [
        f'{test_acc_iter5:.4f}',
        f'{test_loss_iter5:.4f}',
        '3 LSTM (160) + LR Scheduler',
        f'{best_epoch_iter5}'
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + comparison_df.to_string(index=False))

improvement = (test_acc_iter5 - test_acc) * 100
print(f"\n{'='*80}")
print(f"ACCURACY IMPROVEMENT: {improvement:+.2f}%")
print(f"RECOMMENDED MODEL: Iteration 5 (Final)")
print(f"{'='*80}")

# Calculate macro F1 scores
macro_f1_iter1 = f1_score(test_labels_np, test_preds, average='macro')
macro_f1_iter5 = f1_score(test_labels_np_iter5, test_preds_iter5, average='macro')

print(f"\nMacro F1 Scores (for imbalanced dataset):")
print(f"  Iteration 1: {macro_f1_iter1:.4f}")
print(f"  Iteration 5: {macro_f1_iter5:.4f}")

# Save final artifacts
print("\n" + "="*80)
print("SAVING ALL ARTIFACTS")
print("="*80)

# Save final model
model_save_path = f'{MODELS_PATH}/sentiment_classifier_final.pt'
torch.save(model_iter5.state_dict(), model_save_path)
print(f"Model saved: {model_save_path}")

# Save vocabulary
vocab_save_path = f'{MODELS_PATH}/vocab.pkl'
with open(vocab_save_path, 'wb') as f:
    pickle.dump(vocab, f)
print(f"Vocabulary saved: {vocab_save_path}")

# Save training history
history_json = {
    'iteration_1': {
        'test_acc': float(test_acc),
        'macro_f1': float(macro_f1_iter1)
    },
    'iteration_5': {
        'test_acc': float(test_acc_iter5),
        'macro_f1': float(macro_f1_iter5),
        'best_epoch': int(best_epoch_iter5)
    },
    'data_source': 'Kaggle: crowdflower/twitter-airline-sentiment',
    'dataset_notes': 'Real data, 14,640 samples. Vocabulary built from training set only (no leakage)'
}

history_save_path = f'{OUTPUTS_PATH}/training_history.json'
with open(history_save_path, 'w') as f:
    json.dump(history_json, f, indent=2)
print(f"Training history saved: {history_save_path}")

print(f"\n{'='*80}")
print("SENTIMENT CLASSIFIER TRAINING COMPLETE!")
print("="*80)
print(f"\nFiles saved to:")
print(f"  Models: {MODELS_PATH}")
print(f"  Outputs: {OUTPUTS_PATH}")

: 